# Modular Scheduling Examples

Build each act as its own `Schedule`, render acts independently for debugging, then stitch them back together with `Schedule.combine(...)`.

This notebook keeps the schedules explicit:

1. `act_1` introduces the scene.
2. `act_2 = act_1.next_act()` continues from the end of `act_1`.
3. `act_3 = act_2.next_act()` finishes the story.
4. `Schedule.combine(...)` turns the acts back into one animation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML

plt.rcParams["animation.html"] = "jshtml"

for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    src_dir = candidate / "src"
    if src_dir.exists():
        sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Could not locate the project's src directory.")

from visualizer import (
    Curve,
    CurveStyle,
    Draw,
    Erase,
    EraseFillBetween,
    FillBetween,
    FillBetweenArea,
    FillStyle,
    Move,
    MoveFillBetween,
    Parallel,
    Schedule,
    Stress,
)


In [ ]:
x = np.linspace(0.0, 1.0, 250)
y_wave = np.abs(np.sin(np.pi * x))
y_square = np.square(y_wave)
baseline = np.zeros_like(x)


## Act 1

Introduce the original curve and its fill. This act is self-contained and can be rendered on its own.

In [ ]:
act_1 = Schedule()
act_1.add(
    Parallel(
        (
            Draw(Curve("wave", x, y_wave, color="#0f766e", linewidth=3.0)),
            FillBetween(
                FillBetweenArea(
                    "wave_fill",
                    x,
                    y_wave,
                    baseline,
                    color="#99f6e4",
                    alpha=0.35,
                    linewidth=0.0,
                )
            ),
        )
    ),
    duration=1.5,
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.2)
anim = act_1.build_animation(fig=fig, ax=ax, fps=24, title="Act 1")
plt.close(fig)
HTML(anim.to_jshtml())


## Act 2

Start from `act_1.final_scene`, pause briefly, then morph the curve and fill into a new shape. Rendering `act_2` alone is useful when you only want to debug that stage.

In [ ]:
act_2 = act_1.next_act()
act_2.pause(0.35)
act_2.add(
    Parallel(
        (
            Move(
                "wave",
                newy=y_square,
                color="#ea580c",
                linewidth=3.0,
            ),
            MoveFillBetween(
                "wave_fill",
                newy1=y_square,
                newy2=baseline,
                color="#fed7aa",
                alpha=0.4,
            ),
        )
    ),
    duration=1.4,
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.2)
anim = act_2.build_animation(fig=fig, ax=ax, fps=24, title="Act 2 Only")
plt.close(fig)
HTML(anim.to_jshtml())


## Act 3 And Combined Schedule

Finish from the end of `act_2`, then combine all three acts back into one validated schedule.

In [ ]:
act_3 = act_2.next_act()
act_3.add(Stress("wave", glow_color="#fb923c", glow_width=10.0), duration=0.8)
act_3.add(
    Parallel(
        (
            CurveStyle("wave", alpha=0.35, color="#9a3412", linewidth=4.0),
            FillStyle("wave_fill", alpha=0.12, color="#fdba74"),
        )
    ),
    duration=0.8,
)
act_3.add(
    Parallel(
        (
            Erase("wave"),
            EraseFillBetween("wave_fill"),
        )
    ),
    duration=1.0,
)

combined_schedule = Schedule.combine(
    [act_1, act_2, act_3],
    validate_initial_scene=True,
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.2)
anim = combined_schedule.build_animation(
    fig=fig,
    ax=ax,
    fps=24,
    title="Combined Acts",
)
plt.close(fig)
HTML(anim.to_jshtml())
